# YOLO26 PPE Training Pipeline

**clone → pull → install → import → baseline (SHO) → sparsity → prune → recovery (distill / finetune)**

Each stage feeds its output checkpoint into the next. One task per cell.

## 1. Clone the latest code

In [ ]:
from pathlib import Path
import sys

GITHUB_URL = 'https://github.com/hopquangdo/yolo-ppe-optimizer.git'
BRANCH = 'main'
REPO_DIR = Path.home() / 'workspaces' / 'yolo-ppe-optimizer'

In [ ]:
if not (REPO_DIR / 'pyproject.toml').exists():
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    !git clone -b {BRANCH} {GITHUB_URL} "{REPO_DIR}"
else:
    print('Repo already present at', REPO_DIR)

## 2. Pull the newest commits

In [ ]:
%cd $REPO_DIR

In [ ]:
!git checkout {BRANCH}
!git pull --ff-only origin {BRANCH}
!git log -1 --oneline

## 3. Install (editable)

In [ ]:
!{sys.executable} -m pip install -e .

## 4. Imports

In [ ]:
import os

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

import torch
from ultralytics import YOLO

print({'cuda': torch.cuda.is_available(), 'device_count': torch.cuda.device_count()})

## 5. Configuration

In [ ]:
# --- paths ---
MODEL = REPO_DIR / 'weights' / 'yolo26n.pt'     # student / starting checkpoint
TEACHER = REPO_DIR / 'weights' / 'yolo26l.pt'   # distillation teacher
DATA = 'construction-ppe.yaml'                  # built-in PPE dataset (auto-downloads ~178 MB)
PROJECT = REPO_DIR / 'runs' / 'ppe-pipeline'
PROJECT.mkdir(parents=True, exist_ok=True)

# --- stage outputs ---
OPTIMIZED_YAML = PROJECT / 'optimized_baseline.yaml'
SPARSITY_WEIGHTS = PROJECT / 'sparsity.pt'
PRUNED_WEIGHTS = PROJECT / 'yolo26_pruned.pt'
RECOVERED_WEIGHTS = PROJECT / 'recovered.pt'    # output of distill / finetune

In [ ]:
# --- hyperparameters ---
IMG_SIZE = 640
BATCH = 16

BASELINE_EPOCHS = 100
SHO_POP, SHO_GMAX, SHO_PROXY_EPOCHS = 6, 3, 3   # SHO search budget

SPARSITY_EPOCHS = 200
SPARSITY_SR = 1e-2                              # BN-gamma L1 strength (Network Slimming)

PRUNE_RATIO = 0.30                              # fraction of channels to drop

# recovery training after prune: 'distill' (teacher yolo26l) or 'finetune' (plain)
RECOVERY_MODE = 'distill'
RECOVERY_EPOCHS = 100
LAMBDA_CWD = 50.0                               # distill only
KD_TEMPERATURE = 4.0                            # distill only

## 6. Train baseline (SHO)

`scripts/sho_search.py` searches augmentation / LR hyperparameters on a short proxy,
then trains the full baseline and writes `optimized_baseline.yaml`.

In [ ]:
SHO_PROJECT = PROJECT / 'sho'
CMD = (
    f'{sys.executable} scripts/sho_search.py'
    f' --model "{MODEL}"'
    f' --data {DATA}'
    f' --pop {SHO_POP} --gmax {SHO_GMAX} --proxy-epochs {SHO_PROXY_EPOCHS}'
    f' --full-epochs {BASELINE_EPOCHS}'
    f' --imgsz {IMG_SIZE} --batch {BATCH}'
    f' --project "{SHO_PROJECT}"'
    f' --output "{OPTIMIZED_YAML}"'
)
!{CMD}

In [ ]:
_hits = list(SHO_PROJECT.rglob('weights/best.pt'))
assert _hits, f'baseline checkpoint not found under {SHO_PROJECT}'
BASELINE_WEIGHTS = max(_hits, key=lambda p: p.stat().st_mtime)
print('Baseline checkpoint:', BASELINE_WEIGHTS)

## 7. Train sparsity

`scripts/train_sparsity.py` — BN-gamma L1 regularization so channels become prunable.
Attaches the `_bn_gamma_stats` callback; `sr` decays over training. Watch the
`[BN gamma]` log per epoch.

In [ ]:
CMD = (
    f'{sys.executable} scripts/train_sparsity.py'
    f' --weights "{BASELINE_WEIGHTS}"'
    f' --data {DATA}'
    f' --epochs {SPARSITY_EPOCHS} --patience {SPARSITY_EPOCHS}'
    f' --batch {BATCH} --imgsz {IMG_SIZE}'
    f' --sr {SPARSITY_SR}'
    f' --project "{PROJECT}" --name sparsity'
)
!{CMD}

In [ ]:
_hits = list((PROJECT / 'sparsity').rglob('weights/best.pt'))
_hits += list((PROJECT / 'sparsity').rglob('weights/last.pt'))
assert _hits, 'sparsity checkpoint not found'
_src = max(_hits, key=lambda p: p.stat().st_mtime)
SPARSITY_WEIGHTS.write_bytes(_src.read_bytes())
print('Sparsity checkpoint:', SPARSITY_WEIGHTS)

## 8. Prune

Structured channel pruning driven by the BN-gamma masks. Produces `yolo26_pruned.pt`
(a `DetectionModelPruned` plus its `maskbndict`).

In [ ]:
CMD = (
    f'{sys.executable} scripts/prune.py'
    f' --weights "{SPARSITY_WEIGHTS}"'
    f' --model-size n'
    f' --prune-ratio {PRUNE_RATIO}'
    f' --save-dir "{PROJECT}"'
)
!{CMD}

In [ ]:
assert PRUNED_WEIGHTS.exists(), f'pruned checkpoint missing: {PRUNED_WEIGHTS}'
print('Pruned checkpoint:', PRUNED_WEIGHTS)

## 9. Recovery training (distill or finetune)

Two branches after prune, selected by `RECOVERY_MODE`:

- `'distill'` — `scripts/distill.py`, teacher `yolo26l` -> pruned student. Task loss
  (box + cls on GT) is included alongside the CWD distill loss.
- `'finetune'` — `scripts/finetune.py`, plain supervised fine-tuning, no teacher.

In [ ]:
assert RECOVERY_MODE in ('distill', 'finetune'), RECOVERY_MODE

if RECOVERY_MODE == 'distill':
    assert TEACHER.exists(), f'teacher checkpoint missing: {TEACHER}'
    CMD = (
        f'{sys.executable} scripts/distill.py'
        f' --teacher "{TEACHER}"'
        f' --student-checkpoint "{PRUNED_WEIGHTS}"'
        f' --data {DATA}'
        f' --epochs {RECOVERY_EPOCHS} --batch {BATCH}'
        f' --lambda-cwd {LAMBDA_CWD} --temperature {KD_TEMPERATURE}'
        f' --project "{PROJECT}" --name {RECOVERY_MODE}'
    )
else:
    CMD = (
        f'{sys.executable} scripts/finetune.py'
        f' --weights "{PRUNED_WEIGHTS}"'
        f' --data {DATA}'
        f' --epochs {RECOVERY_EPOCHS} --batch {BATCH} --imgsz {IMG_SIZE}'
        f' --project "{PROJECT}" --name {RECOVERY_MODE}'
    )
print(CMD)

In [ ]:
!{CMD}

In [ ]:
_hits = list((PROJECT / RECOVERY_MODE).rglob('weights/best.pt'))
_hits += list((PROJECT / RECOVERY_MODE).rglob('weights/last.pt'))
assert _hits, f'{RECOVERY_MODE} checkpoint not found'
_src = max(_hits, key=lambda p: p.stat().st_mtime)
RECOVERED_WEIGHTS.write_bytes(_src.read_bytes())
print('Recovered checkpoint:', RECOVERED_WEIGHTS)

## 10. Summary

In [ ]:
for label, path in [
    ('baseline', BASELINE_WEIGHTS),
    ('sparsity', SPARSITY_WEIGHTS),
    ('pruned', PRUNED_WEIGHTS),
    (RECOVERY_MODE, RECOVERED_WEIGHTS),
]:
    p = Path(path)
    print(f'{label:10s} {"OK" if p.exists() else "MISSING":8s} {p}')